![IITIS](pictures/logoIITISduze.png)

# Wykorzystanie procesorów graficznych (GPU) w algorytmach heurystycznych

wykorzystamy paczkę numba (można też użyć CuPy wy bezpośrednio użwyać CUDA)

In [1]:
# funkcje pomocniczne

import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from typing import Optional

test_pegasus = os.path.join("instancje", "Pegasus", "P4_CBFM-P.txt")  # E = -469.0
full_pegasus = os.path.join("instancje", "Pegasus", "P16_CBFM-P.txt")  # E = -12772.0



def read_instance(path: os.PathLike):
    df = pd.read_csv(path, sep=" ", header=None, comment="#", names=["i", "j", "value"])

    n = max(df[["i", "j"]].max())
    h = np.zeros(n)
    J = np.zeros((n, n))
    
    for row in df.itertuples():
        if row.i == row.j:
            h[row.i - 1] = row.value
        elif row.i > row.j:
            J[row.j - 1, row.i - 1] = row.value  # by zachować górnotrójkątność
        else:
            J[row.i - 1, row.j - 1] = row.value
            
    return J, h


def dwave_conv_to_minus_half_convention(J: np.ndarray, h: np.ndarray):
    n = len(h)
    herminian_matrix = np.zeros((n, n))

    # de facto wyciągamy -1/2 przed macierz i zamieniamy ją na hermitowską
    for i in range(n):
        for j in range(i + 1, n):
            J_ij = J[i, j]
            herminian_matrix[i, j] = -J_ij
            herminian_matrix[j, i] = -J_ij

    x = np.random.choice([-1, 1], size=n)
    assert np.array_equal(-2 * x @ J @ x.T, x @ herminian_matrix @ x.T)
    assert np.array_equal(herminian_matrix.T, herminian_matrix)  # wszystkie macierze są rzeczywiste

    new_external_fields = -1 * h
    return herminian_matrix, new_external_fields


In [1]:
# Parrarel annealing
# konwencja: J górnotrójkątna, plusy

def calculate_gradient_pararell(J: np.ndarray, h: np.ndarray, x: np.ndarray, state: np.ndarray, 
lambda_t: float) -> np.ndarray:
    _, num_trajectories = x.shape
    return J @ state + J.T @ state + np.stack([h] * num_trajectories, axis=1) + lambda_t * x


def calculate_energy_parrarel(J: np.ndarray, h: np.ndarray, state: np.ndarray, convention: str = "PLUS"):
    n, _ = J.shape
    if convention == "PLUS":
        # Zakładamy, że J jest górnotrójkątna z zerami na przekątnych
        return np.sum(state * ( J @ state + h.reshape(n, 1)), axis=0)
    else:
        return np.sum(state * ( -1/2 * J @ state - h.reshape(n, 1)), axis=0)


def get_best_solution(states, energies):
    best_energy = np.min(energies)
    best_energy_index = np.argmin(energies)
    best_state = states[:, best_energy_index]  # Stany są w kolumnach
    return best_state, best_energy


def parrarel_annealing_multiple_trajectores(J, h, step_size: float, lambda_t_max: float, num_steps: int, num_trajectories: int,
                       schedule: Optional[np.ndarray] = None):
    n = len(h)
    x = np.zeros((n, num_trajectories))  # stan podstawowy dla H_innit = sum(x**2)
    momentum = np.zeros((n, num_trajectories))
    state = np.random.choice([-1, 1], size=(n, num_trajectories))  # losowy stan początkowy

    if schedule is None:
        schedule = np.linspace(lambda_t_max, 0, num=num_steps)

    for k in tqdm(range(num_steps), desc="wyżarzanie równoległe"):
        lambda_t = schedule[k]
        gradient = calculate_gradient_pararell(J, h, x, state, lambda_t)
        momentum = (1 - step_size) * momentum - step_size * gradient
        momentum = np.clip(momentum, -1, 1)
        x += momentum
        x = np.clip(x, -1, 1)
        state = np.sign(x)

    return state, calculate_energy_parrarel(J, h, state, num_trajectories)

In [2]:
import cupy as cp

def calculate_gradient_gpu_naive(J, h, x, state, lambda_t):
    gradient = cp.matmul(cp.multiply(-1, J), state) -  h + cp.multiply(lambda_t, x)
    return gradient

def calculate_energy_gpu_naive(J: np.ndarray, h: np.ndarray, state: np.ndarray):
    # Zakładamy, że J jest hermitowska z czynnikiem 1/2
    n, _ = J.shape
    A = cp.multiply(-1/2, J)
    B = cp.matmul(A, state) - h.reshape(n, 1)
    C = cp.multiply(state, B)
    return cp.sum(C, axis=0)



def parrarel_annealing_gpu_naive(J, h, step_size: float, lambda_t_max: float, num_steps: int, num_trajectories: int,
                       schedule: Optional[np.ndarray] = None, dtype = None):
    n = len(h)
    x = cp.zeros((n, num_trajectories))  # stan podstawowy dla H_innit = sum(x**2)
    momentum = cp.zeros((n, num_trajectories))
    state = cp.random.choice([-1, 1], size=(n, num_trajectories))  # losowy stan początkowy

    h_parrarel = cp.stack([h] * num_trajectories, axis=1)
    beta = cp.float32(1 - step_size)

    if schedule is None:
        schedule = cp.linspace(lambda_t_max, 0, num=num_steps)

    for k in tqdm(range(num_steps), desc="wyżarzanie równoległe GPU"):
        lambda_t = schedule[k]
        gradient = calculate_gradient_gpu_naive(J, h_parrarel, x, state, lambda_t)
        momentum = cp.multiply(beta, momentum) - cp.multiply(step_size, gradient)
        momentum = cp.clip(momentum, -1, 1)
        x = cp.clip(x + momentum, -1, 1)  
        state = cp.sign(x)


    return state, calculate_energy_gpu_naive(J, h, state)


J, h = read_instance(test_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)
J = cp.asarray(J)
h = cp.asarray(h)


states, energies = parrarel_annealing_gpu_naive(J, h, step_size=0.01, lambda_t_max=10, num_steps=1000, num_trajectories=32)
print(min(energies))

wyżarzanie równoległe GPU: 100%|██████████| 1000/1000 [00:00<00:00, 2737.85it/s]

-469.0


In [ ]:
# test na pełnym pegazie
J, h = read_instance(full_pegasus)
J = cp.asarray(J)
h = cp.asarray(h)

states, energies = parrarel_annealing_gpu_naive(J, h, step_size=0.01, lambda_t_max=10, num_steps=10000, num_trajectories=2**10)
print(min(energies))

## Ilustracja rozłożenia w pamięci

In [14]:
# własny kernel

from numba import cuda, float64, float32, int8
from typing import Optional
import numpy as np
import cupy as cp
from cupy.cublas import gemm


# ustawienia
dtype = float32  # typ elementów na jakich działamy
threadsperblock = 256  # Ilość wątków w bloku, musi być określone wcześniej podczas kompilacji jeżeli chcemy używać shared memory 
tilesize = 32


@cuda.jit
def algorithm_step(A, h, x, lambda_t, momentum, beta, alpha, G, M, S):

    # dane dostępne dla każdego wątku w bloku, ograniczona ilość pamięci
    sX_col = cuda.shared.array(shape=threadsperblock, dtype=dtype)   
    sH = cuda.shared.array(shape=threadsperblock, dtype=dtype)
    sA = cuda.shared.array(shape=threadsperblock, dtype=dtype)
    sM = cuda.shared.array(shape=threadsperblock, dtype=dtype)


    ti = cuda.threadIdx.x  # pojedyńczy element w kolumnie
    col = cuda.blockIdx.x  # każdy blok zajmuje się jedną kolumną (trajektorią)
    y = cuda.blockIdx.y  # ewentualne dodatkowe bloki na kolumne

    # gdzie "globalnie" jesteśmy w macierzy
    global_row = ti + y * threadsperblock

    temp1 = dtype(0.)
    temp2 = dtype(0.)
    temp3 = dtype(0.)
    temp4 = dtype(0.)

    if global_row < x.shape[0]:
        sX_col[ti] = x[global_row, col]
        sH[ti] = h[global_row]
        sA[ti] = A[global_row, col]
        sM[ti] = momentum[global_row, col]
    cuda.syncthreads()  # wszystkie wątki widzą dane

    if global_row < x.shape[0]:
        temp1 = -1 * sA[ti] - sH[ti] + sX_col[ti] * lambda_t  # gradient
        temp2 = sM[ti] * beta - alpha * temp1  # momentum
        temp3 = max(-1, min(1, temp2)) # przycinanie momentum
        temp4 = max(-1, min(1, sX_col[ti] + temp3)) # przycinanie x + momentum
        
        M[global_row, col] = temp3  # nowe momentum
        G[global_row, col] = temp4  # nowy x
        # Rzutowanie stanu
        if temp4 > 0:
            S[global_row, col] = 1.
        elif temp4 < 0:
            S[global_row, col] = -1.
        # Niewypełniona macierz automatycznie daje 0 w elementach których nie wskarzemy


@cuda.jit
def sum_state(J, state, A):

    sJ = cuda.shared.array(shape=threadsperblock, dtype=dtype)
    sS = cuda.shared.array(shape=threadsperblock, dtype=dtype)

    ti = cuda.threadIdx.x  # pojedyńczy element w kolumnie
    col = cuda.blockIdx.x  # każdy blok zajmuje się jedną kolumną (trajektorią)
    bpg = J.shape[0] // threadsperblock + 1 # ilośc dodatkowych blokow

    for row in range(J.shape[0]):
        temp = dtype(0.)
        for b in range(bpg + 1):
            i = ti + b * threadsperblock
            if i < state.shape[0]:
                sJ[ti] = J[row, i]
                sS[ti] = state[i, col]

            cuda.syncthreads()  # wszystkie wątki widzą dane

            if i < state.shape[0]:
                for k in range(threadsperblock):
                    temp += sJ[k] * sS[k]

            cuda.syncthreads()  # wszystkie wątki skonczyly obliczenia
        A[row, col] = temp

        


def calculate_energy_gpu_naive(J: cp.ndarray, h: cp.ndarray, state: cp.ndarray):
    # Zakładamy, że J jest hermitowska z czynnikiem 1/2
    n, _ = J.shape
    A = cp.multiply(-1/2, J)
    B = cp.matmul(A, state) - h.reshape(n, 1)
    C = cp.multiply(state, B)
    return cp.sum(C, axis=0)



def parrarel_annealing_gpu(J, h, step_size: float, lambda_t_max: float, num_steps: int, num_trajectories: int,
                       schedule: Optional[np.ndarray] = None):
    n = len(h)
    x = cp.zeros((n, num_trajectories))  # stan podstawowy dla H_innit = sum(x**2)
    momentum = cp.zeros((n, num_trajectories))
    state = cp.random.choice([-1., 1.], size=(n, num_trajectories)) # losowy stan początkowy

    beta = dtype(1 - step_size)

    if schedule is None:
        schedule = np.linspace(lambda_t_max, 0, num=num_steps)

    blockspergrid_x = num_trajectories  # każdy blok zajmuje się trajektorią
    blockspergrid_y = (n + threadsperblock - 1) // threadsperblock  # wystarczająca ilość bloków by pomieścić całą kolumnę 
    blockspergrid = (blockspergrid_x, blockspergrid_y)

    G = cuda.device_array_like(x)
    M = cuda.device_array_like(x)
    S = cuda.device_array((n, num_trajectories))
    A = cuda.device_array_like(x)
    # G = cp.empty_like(x)
    # M = cp.empty_like(x)
    # S = cp.empty_like(state)
    # A = cp.empty_like(x)


  
    for k in tqdm(range(num_steps), desc="wyżarzanie równoległe GPU"):
        lambda_t = schedule[k]
        #fast_matmul[blocks_per_grid, threads_per_block](J, state, A)
        #gemm("N", "N", J, state, A)  # nie wymyślajmy koła od nowa
        sum_state[blockspergrid_x, threadsperblock](J, state, A)
        algorithm_step[blockspergrid, threadsperblock](A, h, x, lambda_t, momentum, beta, step_size, G, M, S)
        momentum = M
        x = G
        #state = cp.asarray(S, dtype=np.float64)

        state = S
        print(A.copy_to_host())
        break
    state = cp.asarray(state)
    return state, calculate_energy_gpu_naive(J, h, state)


In [13]:
16000// 256
for i in range(2):
    print(i)


0
1


In [18]:
J, h = read_instance(test_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)

J = cp.asarray(J, dtype=np.float64)
h = cp.asarray(h)

states, energy = parrarel_annealing_gpu(J, h, step_size=0.01, lambda_t_max=10, num_steps=1000, num_trajectories=32)
print(min(energy))

wyżarzanie równoległe GPU:   0%|          | 0/1000 [00:00<?, ?it/s]

[[-inf   3.  -1. ...  -3.   3.   3.]
 [-inf   2.  -2. ...  -2.   0.   4.]
 [-inf   0.   0. ...   2.   0.   2.]
 ...
 [-inf  -2.   2. ...  -2.   0.   4.]
 [-inf   0.   0. ...  -4.   0.   2.]
 [-inf   2.  -2. ...  -2.   2.  -2.]]
-57.32


In [45]:
J, h = read_instance(full_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)

J = cp.asarray(J)
h = cp.asarray(h)

states, energy = parrarel_annealing_gpu(J, h, step_size=0.01, lambda_t_max=10, num_steps=1000, num_trajectories=2**10)
print(min(energy))

wyżarzanie równoległe GPU:  31%|███▏      | 314/1000 [00:34<01:14,  9.18it/s]

Unexpected exception formatting exception. Falling back to standard exception



Traceback (most recent call last):
  File "/home/tsmierzchalski/anaconda3/envs/szkolenia/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1241282/2780100186.py", line 7, in <module>
    states, energy = parrarel_annealing_gpu(J, h, step_size=0.01, lambda_t_max=10, num_steps=1000, num_trajectories=2**10)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1241282/830663316.py", line 146, in parrarel_annealing_gpu
    algorithm_step[blockspergrid, threadsperblock](A, h, x, lambda_t, momentum, beta, step_size, G, M, S)
  File "/home/tsmierzchalski/anaconda3/envs/szkolenia/lib/python3.12/site-packages/numba_cuda/numba/cuda/dispatcher.py", line 600, in __call__
    min_grid_size = 128
       ^^^^^^^^^^^^^^^^
  File "/home/tsmierzchalski/anaconda3/envs/szkolenia/lib/python3.12/site-pac

In [16]:
# brute force CUDA

from numba import cuda, float32, float64
import cupy as cp

@cuda.jit
def brute_force_gpu():
    ...


    

0


In [ ]:
# Simulated bifurcation GPU naive

def balistic_simulated_bifurcation(J, h, num_steps, time_step, num_trajectories: int, 
                                   a_0: Optional[float] = None, c_0_scaling: Optional[float] = None):
    if a_0 is None:
        a_0 = 1

    N, _ = J.shape
    mean_J = np.sqrt(np.sum(np.square(J)) / (N * (N - 1)) )
    c_0 = 0.5 / (mean_J * sqrt(N))

    if c_0_scaling is not None:
        c_0 *= c_0_scaling


    a = np.linspace(0, a_0, num=num_steps)

    x = np.zeros((N, num_trajectories))
    y = np.random.uniform(-0.1, 0.1, (N, num_trajectories))

    for t in range(num_steps):
        y += (-1 * (a_0 - a[t]) * x + c_0 * J @ x) * time_step  # x(t)
        x += a_0 * y * time_step # x(t + 1)

        x, y = wall(x, y)

    x = np.sign(x)
    return x, calculate_energy_parrarel(J, h, x, "")

In [33]:
# własny kernel

from numba import cuda, float64, float32, int8
from nvmath.device import matmul, Dim3
import numpy as np
import cupy as cp

dtype = float32
threadsperblock = 256
block_dim = Dim3(32, 32, 1)
N = 5
num_trajectories = 2





@cuda.jit
def update_x(X, Y, a_0, time_step, X_new, Y_new):
    sX = cuda.shared.array(shape=threadsperblock, dtype=dtype)
    sY = cuda.shared.array(shape=threadsperblock, dtype=dtype)
    
    ti = cuda.threadIdx.x  # pojedyńczy element w kolumnie
    col = cuda.blockIdx.x  # każdy blok zajmuje się jedną kolumną (trajektorią)
    y = cuda.blockIdx.y  # ewentualne dodatkowe bloki na kolumne

    # gdzie "globalnie" jesteśmy w macierzy
    global_row = ti + y * threadsperblock

    temp1 = dtype(0.)
    if global_row < X.shape[0]:
        sX[ti] = X[global_row, col]
        sY[ti] = Y[global_row, col]
    cuda.syncthreads()  # wszystkie wątki widzą dane

    if global_row < X.shape[0]:
        temp1 = sX[ti] + a_0 * sY[ti] * time_step  # krok x

        # "sciana"
        if abs(temp1) > 1:
            X_new[global_row, col] = max(-1, min(1, temp1))
            Y_new[global_row, col] = 0
        else:
            X_new[global_row, col] = temp1
            Y_new[global_row, col] = sY[ti]


J = cuda.to_device(np.random.uniform(-1, 1, (N, N)))
x = cuda.to_device(np.random.uniform(-1, 1, (N, num_trajectories)))
y = cuda.to_device(np.random.uniform(-1, 1, (N, num_trajectories)))
x_new = cuda.device_array_like(x)
y_new = cuda.device_array_like(x)
a_0 = 1
a_t = 0.1
time_step = 0.25
c_0 = 0.25

beta = -1 * (a_0 - a_t)

blockspergrid_x = num_trajectories  # każdy blok zajmuje się trajektorią
blockspergrid_y = (n + threadsperblock - 1) // threadsperblock  # wystarczająca ilość bloków by pomieścić całą kolumnę 
blockspergrid = (blockspergrid_x, blockspergrid_y)
print(MM.files)
# update_y[1, block_dim](J, x, y, c_0, beta, y_new)
# y = y_new
# update_x[blockspergrid, threadsperblock](x, y, a_0, time_step, x_new, y_new)
# print(x_new.copy_to_host())
# print(y_new.copy_to_host())
# print(y.copy_to_host())

['/tmp/tmpmxyojy92.ltoir']


ModuleNotFoundError: No module named 'accelerate.cuda'

# 